# Lab MQTT — 01. Topic Analysis

Analizza i 4 file `topicdf_*.pkl` forniti dal prof. ed estrae le **liste di topic** che useremo nel client MQTT (notebook `02_mqtt_collector.py`) per sottoscriverci ai broker pubblici **senza usare il wildcard `#` al root** (perché Mosquitto/HiveMQ/EMQX lo proibiscono).

**Output principali:**
- `top_root_topics.csv` — top root + frequenza di broker
- `top_level_words.csv` — parole più frequenti a qualunque livello
- `subscription_list.json` — la lista finale di **filter topics** che useremo in subscribe (con QoS 2)
- alcuni grafici riassuntivi (depth/length CDF, top root bar chart) in stile *paper*

In [ ]:
import pandas as pd
import re
import json
import gc
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import numpy as np

DATA_DIR = Path("/mnt/user-data/uploads")  # cambia se i .pkl sono altrove
OUT_DIR  = Path("./outputs")
OUT_DIR.mkdir(exist_ok=True)

FILES = sorted(DATA_DIR.glob("topicdf_*.pkl"))
print("Trovati:", FILES)

## 1. Inspezione di base

In [ ]:
# Apriamo solo il primo file per capire la struttura
df = pd.read_pickle(FILES[0], compression="gzip")
print("Shape:", df.shape)
print("Colonne:", list(df.columns))
df.head(2)

In [ ]:
# Ogni riga = 1 broker. La colonna topic_list è una STRINGA con un dict Python
# del tipo:  { "<topic>": {"total": N, "topic_depth": K, "topic_length": L, ...}, ... }
# Il dict può essere enorme (alcuni broker hanno >500k topic), per questo non usiamo
# ast.literal_eval su tutto il dataset: usiamo regex per estrarre solo ciò che ci serve.
raw = df.iloc[1]['topic_list']
print("Lunghezza stringa:", len(raw))
print("Primi 400 char:\n", raw[:400])

## 2. Estrazione efficiente con regex

Per evitare problemi di memoria (alcune righe sono >80 MB di stringa) estraiamo direttamente i campi `topic_depth`, `topic_length`, `total` e i nomi dei topic con regex.

In [ ]:
# Pattern: i top-level key del dict sono nella forma   '<topic>': {'total': ...
key_pat    = re.compile(r"'([^']+)': \{'total':")
depth_pat  = re.compile(r"'topic_depth': (\d+)")
length_pat = re.compile(r"'topic_length': (\d+)")

all_topics_counter      = Counter()  # <topic completo>: # occorrenze (su quanti broker compare)
root_by_broker_counter  = Counter()  # <root>: # broker distinti
level_by_broker_counter = Counter()  # <parola di un livello>: # broker distinti
depth_counter  = Counter()           # depth -> #topic
length_counter = Counter()           # length -> #topic

broker_topic_counts = []  # quanti topic ha ciascun broker
total_brokers = 0

for f in FILES:
    df = pd.read_pickle(f, compression="gzip")
    print(f"  {f.name}: {len(df)} broker")
    for raw in df['topic_list']:
        total_brokers += 1
        topic_names = key_pat.findall(raw)
        broker_topic_counts.append(len(topic_names))
        broker_roots, broker_levels = set(), set()
        for t in topic_names:
            all_topics_counter[t] += 1
            if t.startswith('$SYS'):
                continue  # ignoriamo i system topic come da paper
            non_empty = [p for p in t.split('/') if p]
            if non_empty:
                broker_roots.add(non_empty[0])
            for p in non_empty:
                broker_levels.add(p)
        for r in broker_roots:  root_by_broker_counter[r] += 1
        for lv in broker_levels: level_by_broker_counter[lv] += 1
        # depth/length per ogni topic
        for d in depth_pat.findall(raw):  depth_counter[int(d)] += 1
        for l in length_pat.findall(raw): length_counter[int(l)] += 1
    del df
    gc.collect()

print(f"\nTotale broker analizzati: {total_brokers}")
print(f"Topic distinti totali:    {len(all_topics_counter):,}")
print(f"Root distinti (no \\$SYS): {len(root_by_broker_counter):,}")
print(f"Parole-livello distinte: {len(level_by_broker_counter):,}")

## 3. Top root topics (paper-style: % di broker su cui appaiono)

In [ ]:
TOP_N = 30
rows = [(name, c, 100*c/total_brokers) for name, c in root_by_broker_counter.most_common(TOP_N)]
top_roots = pd.DataFrame(rows, columns=['root', 'n_brokers', 'pct_brokers'])
top_roots.to_csv(OUT_DIR / "top_root_topics.csv", index=False)
top_roots

In [ ]:
# Bar chart in stile Figura 7 del paper
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(top_roots['root'][::-1], top_roots['pct_brokers'][::-1])
for i, v in enumerate(top_roots['pct_brokers'][::-1]):
    ax.text(v + 0.05, i, f"{v:.1f}", va='center')
ax.set_xlabel('Frequency of Brokers [%]')
ax.set_title('Top Root Topics in dataset (excl. $SYS)')
ax.grid(axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_top_roots.png", dpi=140)
plt.show()

## 4. Top parole-livello (utili per fare wildcard `+`)

In [ ]:
rows = [(name, c, 100*c/total_brokers) for name, c in level_by_broker_counter.most_common(TOP_N)]
top_levels = pd.DataFrame(rows, columns=['level_word', 'n_brokers', 'pct_brokers'])
top_levels.to_csv(OUT_DIR / "top_level_words.csv", index=False)
top_levels

## 5. Distribuzione di topic depth e length (CDF)

In [ ]:
def cdf_from_counter(counter):
    items = sorted(counter.items())
    xs = np.array([k for k,_ in items])
    ys = np.cumsum([c for _,c in items], dtype=float)
    ys /= ys[-1]
    return xs, ys

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x, y = cdf_from_counter(depth_counter)
axes[0].step(x, y, where='post')
axes[0].set_xscale('log')
axes[0].set_xlabel('Topic size (number of levels)')
axes[0].set_ylabel('CDF')
axes[0].set_title('Topic depth CDF')
axes[0].grid(True, which='both', alpha=0.3)

x, y = cdf_from_counter(length_counter)
axes[1].step(x, y, where='post')
axes[1].set_xscale('log')
axes[1].set_xlabel('Topic length (bytes)')
axes[1].set_ylabel('CDF')
axes[1].set_title('Topic length CDF')
axes[1].grid(True, which='both', alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / "fig_topic_depth_length.png", dpi=140)
plt.show()

def percentiles_from_counter(counter, qs=(25,50,75,90,95,99)):
    total = sum(counter.values())
    out = {}
    cum = 0
    targets = {q: int(total*q/100) for q in qs}
    for k in sorted(counter):
        cum += counter[k]
        for q in list(targets):
            if cum >= targets[q]:
                out[q] = k
                del targets[q]
        if not targets:
            break
    return out

print("Depth percentiles :", percentiles_from_counter(depth_counter))
print("Length percentiles:", percentiles_from_counter(length_counter))

## 6. Costruzione della *subscription list*

Vincolo del prof.: **niente `#` al root**. Quindi niente subscribe a `#`. Possiamo invece:
1. Sottoscriverci ai **root topic più frequenti** con `<root>/#` (wildcard solo a partire dal secondo livello → permesso sui broker pubblici).
2. Aggiungere alcuni filtri *single-level* misti tipo `+/<word>/#` per testare strutture intermedie.

Costruiamo una lista di ~60 topic-filter.

In [ ]:
# Filtra root "sospetti" (numerici puri o cose ovviamente non standard)
def is_clean(token):
    return bool(re.match(r'^[A-Za-z][A-Za-z0-9_\-]{1,30}$', token))

ROOTS_TO_USE = [r for r,_ in root_by_broker_counter.most_common(80) if is_clean(r)][:50]
LEVELS_TO_USE = [w for w,_ in level_by_broker_counter.most_common(50) if is_clean(w)][:15]

subscription_filters = []
# 1) root wildcard
for r in ROOTS_TO_USE:
    subscription_filters.append(f"{r}/#")

# 2) qualche pattern misto (es. +/state, +/status, +/config, +/event ...)
for w in ["state", "status", "config", "event", "sensor", "data", "temperature", "command"]:
    if w in LEVELS_TO_USE:
        subscription_filters.append(f"+/{w}")
        subscription_filters.append(f"+/+/{w}")

# 3) qualche topic specifico molto comune (Tasmota / HomeAssistant / Zigbee2MQTT)
subscription_filters += [
    "tele/+/+", "stat/+/+", "cmnd/+/+",          # Tasmota
    "homeassistant/#",                           # Home Assistant discovery
    "zigbee2mqtt/+", "zigbee2mqtt/+/+",          # Zigbee2MQTT
    "shellies/+/+/+",                            # Shelly devices
    "$SYS/broker/version",                       # solo come info diagnostica
    "$SYS/broker/clients/connected",
]

# Deduplica mantenendo l'ordine
seen = set(); subs = []
for f in subscription_filters:
    if f not in seen:
        seen.add(f); subs.append(f)

print(f"Numero di filtri di subscribe generati: {len(subs)}")
for s in subs[:25]:
    print(" ", s)
print("  ...")

In [ ]:
# Esporta la lista per il collector. QoS=2 (richiesto dal prof) per ogni subscribe.
payload = {
    "qos": 2,
    "topics": subs
}
with open(OUT_DIR / "subscription_list.json", "w") as fh:
    json.dump(payload, fh, indent=2)
print("Salvato:", OUT_DIR / "subscription_list.json")

## 7. Sanity check finale

In [ ]:
import pandas as pd
summary = pd.Series({
    "# brokers analyzed":  total_brokers,
    "# distinct topics":   len(all_topics_counter),
    "# distinct roots":    len(root_by_broker_counter),
    "median topic depth":  percentiles_from_counter(depth_counter)[50],
    "median topic length": percentiles_from_counter(length_counter)[50],
    "# subscribe filters": len(subs),
})
print(summary.to_string())